In [16]:
import lightgbm as lgb
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import pandas as pd

## Data Exploration

In [17]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Dataset shape
df_shape = df.shape
df_height = df_shape[0]
df_width = df_shape[1]


cols = df.columns.tolist()


num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Dataset Shape: {df_shape}")
print(f"Rows: {df_height}")
print(f"Columns: {df_width}")
print("\nNumeric Columns:")
print(num_cols)
print("\nCategorical Columns:")
print("\nFirst 5 Rows:")

Dataset Shape: (7043, 21)
Rows: 7043
Columns: 21

Numeric Columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical Columns:

First 5 Rows:


/tmp/ipykernel_51801/3214464946.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [18]:

print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [19]:
print(f"Null (%) for each column:\n{(df.isnull().mean()*100).round(2)}")

Null (%) for each column:
customerID          0.0
gender              0.0
SeniorCitizen       0.0
Partner             0.0
Dependents          0.0
tenure              0.0
PhoneService        0.0
MultipleLines       0.0
InternetService     0.0
OnlineSecurity      0.0
OnlineBackup        0.0
DeviceProtection    0.0
TechSupport         0.0
StreamingTV         0.0
StreamingMovies     0.0
Contract            0.0
PaperlessBilling    0.0
PaymentMethod       0.0
MonthlyCharges      0.0
TotalCharges        0.0
Churn               0.0
dtype: float64


## Data Processing

Xóa ID khách hàng

In [20]:
df.drop("customerID", axis=1, inplace=True)

Chuyển TotalCharges sang số

In [21]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)



Target

In [22]:
df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

Encode Categorical

In [23]:
categorical_cols = df.select_dtypes(
    include=["object"]
).columns

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

/tmp/ipykernel_51801/1892738396.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


Tách X và y

In [24]:
X = df.drop("Churn", axis=1).values
y = df["Churn"].values

Tự viết Train Test Split

In [25]:
def train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42):

    np.random.seed(random_state)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(
        len(X)*(1-test_size)
    )

    train_idx = indices[:split]
    test_idx = indices[split:]

    return (
        X[train_idx],
        X[test_idx],
        y[train_idx],
        y[test_idx]
    )

## Huấn luyện LightGBM

Cross Validation

In [26]:
def stratified_kfold(y, k=5, random_state=42):
    np.random.seed(random_state)

    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    np.random.shuffle(idx_0)
    np.random.shuffle(idx_1)

    folds_0 = np.array_split(idx_0, k)
    folds_1 = np.array_split(idx_1, k)

    folds = []

    for i in range(k):
        test_idx = np.concatenate([
            folds_0[i],
            folds_1[i]
        ])

        train_idx = np.concatenate([
            np.concatenate([folds_0[j] for j in range(k) if j != i]),
            np.concatenate([folds_1[j] for j in range(k) if j != i])
        ])

        folds.append((train_idx, test_idx))

    return folds

In [27]:
folds = stratified_kfold(y, k=5)

In [28]:
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

Confusion Matrix

In [29]:
def confusion_matrix(
        y_true,
        y_pred):

    tp = np.sum(
        (y_true==1) &
        (y_pred==1)
    )

    tn = np.sum(
        (y_true==0) &
        (y_pred==0)
    )

    fp = np.sum(
        (y_true==0) &
        (y_pred==1)
    )

    fn = np.sum(
        (y_true==1) &
        (y_pred==0)
    )

    return tp,tn,fp,fn

Accuracy

In [30]:
def accuracy_score(
        y_true,
        y_pred):

    return np.mean(
        y_true == y_pred
    )

Precision

In [31]:
def precision_score(
        y_true,
        y_pred):

    tp,tn,fp,fn = \
    confusion_matrix(
        y_true,
        y_pred
    )

    return tp/(tp+fp)

Recall

In [32]:
def recall_score(
        y_true,
        y_pred):

    tp,tn,fp,fn = \
    confusion_matrix(
        y_true,
        y_pred
    )

    return tp/(tp+fn)

F1 Score

In [33]:
def f1_score(
        y_true,
        y_pred):

    precision = precision_score(
        y_true,
        y_pred
    )

    recall = recall_score(
        y_true,
        y_pred
    )

    return (
        2*precision*recall
        /
        (precision+recall)
    )

In [34]:

for train_idx, test_idx in folds:

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc_scores.append(
        accuracy_score(y_test, y_pred)
    )

    precision_scores.append(
        precision_score(y_test, y_pred)
    )

    recall_scores.append(
        recall_score(y_test, y_pred)
    )

    f1_scores.append(
        f1_score(y_test, y_pred)
    )

[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000891 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 637
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265353 -> initscore=-1.018328
[LightGBM] [Info] Start training from score -1.018328
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000370 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 637
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 30
[LightGBM] [Info] [binary:

Print 

In [35]:
print("Average Accuracy :", np.mean(acc_scores))
print("Average Precision:", np.mean(precision_scores))
print("Average Recall   :", np.mean(recall_scores))
print("Average F1 Score :", np.mean(f1_scores))

Average Accuracy : 0.7948308745232573
Average Precision: 0.6370782697498149
Average Recall   : 0.5302289572909349
Average F1 Score : 0.5783598398087982
